In [1]:
# ==========================================================
# 🚀 SAR → CNN → TM → INTERPRETATION (WITH PRED PRINT)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit
import pandas as pd

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ✅ CLASS NAMES (NEW)
classes = sorted(os.listdir(TRAIN_DIR))

# ================= DATASET =================

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        self.classes = sorted(os.listdir(folder))

        for label, cname in enumerate(self.classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        img = np.stack([img]*3, axis=0)

        return torch.tensor(img), torch.tensor(label)

# ================= LOAD =================

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(classes)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID

                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):

                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std  / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = [
                                    "mean_rel_low","mean_rel_mid","mean_rel_high",
                                    "std_rel_low","std_rel_high",
                                    "cv_low","cv_high",
                                    "max_rel_mean","max_rel_std"
                                ]

                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)

    acc = accuracy_score(y_test, preds)
    history.append([epoch+1, acc])

    print(f"Epoch {epoch+1} → {acc:.4f}")

    # ✅ PRINT ONLY FINAL EPOCH
    if epoch == TM_EPOCHS - 1:
        print("\n🔥 Sample Predictions (First 10):")
        for i in range(10):
            print(
                f"Sample {i} → Predicted: {classes[preds[i]]} "
                f"| True: {classes[y_test[i]]}"
            )

# SAVE RESULTS
df = pd.DataFrame(history, columns=["epoch", "accuracy"])
df.to_excel("patch_4x4_results_mstar.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ================= HEATMAP =================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_dataset[idx]
    img_np = img[0].cpu().numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape
    patch_h = h // PATCH_GRID
    patch_w = w // PATCH_GRID

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*patch_h:(i+1)*patch_h,
            j*patch_w:(j+1)*patch_w
        ]

        if patch.size == 0:
            continue

        patch_norm = patch / (patch.max() + 1e-6)
        weight = 2.0 if "max" in info["type"] else 1.0

        heatmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w] += patch_norm * weight

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)

    heatmap_norm = cv2.normalize(
        heatmap, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(
        heatmap_norm, cv2.COLORMAP_JET
    )

    cv2.imwrite(os.path.join(save_dir, f"heatmap_mstar_{idx}.png"), heatmap_color)

    img_uint8 = (img_np * 255).astype(np.uint8)
    img_color = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(img_color, 0.6, heatmap_color, 0.4, 0)

    cv2.imwrite(os.path.join(save_dir, f"overlay_mstar_{idx}.png"), overlay)

    print(f"✅ Saved heatmap + overlay for {idx}")

print("Generating interpretations...")

for i in range(3):
    generate_heatmap(i)

Training CNN...
✅ CNN Done
Extracting features...
Training TM...
2026-04-22 14:00:59,588 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-22 14:00:59,589 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-22 14:00:59,590 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-22 14:00:59,591 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-22 14:00:59,592 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-22 14:01:00,402 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/t

In [1]:
# ==========================================================
# 🚀 SAR → CNN → TM → INTERPRETATION (WITH INTERMEDIATE HEATMAP)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit

# ================= CONFIG =================

BASE_DIR = "MSTAR-10-Classes"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 32
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= DATASET =================

class SARDataset(Dataset):
    def __init__(self, folder, limit=None):
        self.samples = []
        classes = sorted(os.listdir(folder))

        for label, cname in enumerate(classes):
            paths = glob(os.path.join(folder, cname, "*"))

            if limit:
                paths = random.sample(paths, min(limit, len(paths)))

            for p in paths:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        img = np.stack([img]*3, axis=0)

        return torch.tensor(img), torch.tensor(label)

# ================= LOAD =================

train_dataset = SARDataset(TRAIN_DIR, TRAIN_PER_CLASS)
test_dataset  = SARDataset(TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(os.listdir(TRAIN_DIR))

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    
                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID
                    
                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):
                    
                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std  / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = [
                                    "mean_rel_low","mean_rel_mid","mean_rel_high",
                                    "std_rel_low","std_rel_high",
                                    "cv_low","cv_high",
                                    "max_rel_mean","max_rel_std"
                                ]

                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

import pandas as pd

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)
    acc = accuracy_score(y_test, preds)

    history.append([epoch+1, acc])
    print(f"Epoch {epoch+1} → {acc:.4f}")

# SAVE TO EXCEL
df = pd.DataFrame(history, columns=["epoch", "accuracy"])
df.to_excel("patch_4x4_results_mstar.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ==========================================================
# 🔥 INTERPRETATION MODULE
# ==========================================================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_dataset[idx]
    img_np = img[0].cpu().numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape
    patch_h = h // PATCH_GRID  # 16
    patch_w = w // PATCH_GRID  # 16

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*patch_h:(i+1)*patch_h,
            j*patch_w:(j+1)*patch_w
        ]
        if patch.size == 0:
            continue
        patch_norm = patch / (patch.max() + 1e-6)

        weight = 2.0 if "max" in info["type"] else 1.0

        # Use patch_h/patch_w instead of hardcoded 32
        heatmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w] += patch_norm * weight


    # 🔥 INTERMEDIATE HEATMAP
    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)

    heatmap_norm = cv2.normalize(
        heatmap, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(
        heatmap_norm, cv2.COLORMAP_JET
    )

    # SAVE PURE HEATMAP
    cv2.imwrite(os.path.join(save_dir, f"heatmap_mstar_{idx}.png"), heatmap_color)

    # OPTIONAL: OVERLAY
    img_uint8 = (img_np * 255).astype(np.uint8)
    img_color = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(img_color, 0.6, heatmap_color, 0.4, 0)

    cv2.imwrite(os.path.join(save_dir, f"overlay_mstar_{idx}.png"), overlay)

    print(f"✅ Saved heatmap + overlay for {idx}")

# ================= RUN =================

print("Generating interpretations...")

for i in range(3):
    generate_heatmap(i)

Training CNN...
✅ CNN Done
Extracting features...
Training TM...
2026-04-14 05:23:15,989 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-14 05:23:15,991 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-14 05:23:15,993 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-14 05:23:15,994 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-14 05:23:15,995 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-14 05:23:16,420 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/t

In [1]:
# ==========================================================
# 🚀 INDIAN PINES → CNN → TM → INTERPRETATION (FINAL)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit
import pandas as pd

# ================= CONFIG =================

PATCH_SIZE = 11
TRAIN_PER_CLASS = 200
BATCH_SIZE = 32
CNN_EPOCHS = 40

CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD DATA =================

print("Loading Indian Pines dataset...")

X = np.load("indianpinearray.npy")  # (145,145,200)
y = np.load("IPgt.npy")             # (145,145)

# ================= MATCH SAR INPUT =================

X_gray = np.mean(X, axis=2)
X_gray = (X_gray - X_gray.min()) / (X_gray.max() + 1e-6)
X = np.stack([X_gray]*3, axis=2)  # (H,W,3)

# ================= PATCH CREATION =================

def create_classwise_samples(X, y, patch_size, limit_per_class):
    margin = patch_size // 2
    padded = np.pad(X, ((margin,margin),(margin,margin),(0,0)), mode='reflect')

    class_samples = {}

    for i in range(margin, X.shape[0] + margin):
        for j in range(margin, X.shape[1] + margin):

            label = y[i-margin, j-margin]
            if label == 0:
                continue

            patch = padded[i-margin:i+margin+1, j-margin:j+margin+1]
            patch = patch.transpose(2,0,1)

            label_idx = label - 1

            if label_idx not in class_samples:
                class_samples[label_idx] = []

            class_samples[label_idx].append(patch)

    X_out, y_out = [], []

    for cls, samples in class_samples.items():
        if len(samples) > limit_per_class:
            samples = random.sample(samples, limit_per_class)

        for patch in samples:
            X_out.append(patch)
            y_out.append(cls)

    return np.array(X_out), np.array(y_out)

print("Creating patches...")

X_data, y_data = create_classwise_samples(
    X, y, PATCH_SIZE, TRAIN_PER_CLASS
)

# ================= SPLIT =================

indices = np.arange(len(X_data))
np.random.shuffle(indices)

split = int(0.8 * len(indices))

train_idx = indices[:split]
test_idx  = indices[split:]

X_train, y_train = X_data[train_idx], y_data[train_idx]
X_test,  y_test  = X_data[test_idx],  y_data[test_idx]

num_classes = len(np.unique(y_data))

# ================= DATASET =================

class HSIDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx])
        )

train_loader = DataLoader(HSIDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(HSIDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    Xf, yf = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    h, w = ch_np.shape

                    ph = max(1, h // PATCH_GRID)
                    pw = max(1, w // PATCH_GRID)

                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):

                            patch = ch_np[
                                i*ph:min((i+1)*ph, h),
                                j*pw:min((j+1)*pw, w)
                            ]

                            if patch.size == 0:
                                continue

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            f = [
                                int(mean/(ch_mean+1e-6) > 0.8),
                                int(mean/(ch_mean+1e-6) > 1.0),
                                int(std/ch_std > 1.0),
                                int(std/(mean+1e-6) > 0.5),
                                int(mx > ch_mean)
                            ]

                            sample_features.extend(f)

                            if not built:
                                for t in range(len(f)):
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            Xf.append(np.array(batch_feats))
            yf.extend(yb.numpy())

    return np.vstack(Xf).astype(np.uint32), np.array(yf, dtype=np.uint32)

print("Extracting features...")

X_train_tm, y_train_tm = extract(train_loader)
X_test_tm,  y_test_tm  = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train_tm, y_train_tm, epochs=1)
    preds = tm.predict(X_test_tm)
    acc = accuracy_score(y_test_tm, preds)

    history.append([epoch+1, acc])
    print(f"Epoch {epoch+1} → {acc:.4f}")

pd.DataFrame(history, columns=["epoch","accuracy"]).to_excel("results_indianpines.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ================= HEATMAP =================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_loader.dataset[idx]
    img_np = img[0].cpu().numpy()

    heatmap = np.zeros_like(img_np, dtype=np.float32)

    active = np.where(X_test_tm[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape

    ph = max(1, h // PATCH_GRID)
    pw = max(1, w // PATCH_GRID)

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*ph:min((i+1)*ph, h),
            j*pw:min((j+1)*pw, w)
        ]

        if patch.size == 0:
            continue

        patch_norm = patch / (patch.max() + 1e-6)
        weight = 2.0 if info["type"] == 4 else 1.0

        heatmap[i*ph:min((i+1)*ph, h), j*pw:min((j+1)*pw, w)] += patch_norm * weight

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)
    heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img_uint8 = (img_np * 255).astype(np.uint8)
    img_color = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)

    overlay = cv2.addWeighted(img_color, 0.6, heatmap_color, 0.4, 0)

    cv2.imwrite(f"{save_dir}/heatmap_indianpines{idx}.png", heatmap_color)
    cv2.imwrite(f"{save_dir}/overlay_indianpines{idx}.png", overlay)

    print(f"Saved heatmap {idx}")

print("Generating heatmaps...")

for i in range(3):
    generate_heatmap(i)

Loading Indian Pines dataset...
Creating patches...
Training CNN...
✅ CNN Done
Extracting features...
Training TM...
2026-04-14 15:00:53,191 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-14 15:00:53,193 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-14 15:00:53,194 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-14 15:00:53,195 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-14 15:00:53,196 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-14 15:00:53,517 - tmu.clause_bank.clause_bank

In [1]:
# ==========================================================
# 🚀 SAMPLE_DATASET → CNN → TM → INTERPRETATION (FIXED)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pandas as pd
import pycuda.autoinit

# ================= CONFIG =================

BASE_DIR = "sample_dataset"

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 64
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TM
CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= PREPARE DATA =================

print("Preparing dataset...")

train_samples = []
test_samples = []

classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    class_path = os.path.join(BASE_DIR, cname)

    if not os.path.isdir(class_path):
        continue

    paths = glob(os.path.join(class_path, "*"))
    random.shuffle(paths)

    train_paths = paths[:TRAIN_PER_CLASS]
    test_paths  = paths[TRAIN_PER_CLASS:]

    for p in train_paths:
        train_samples.append((p, label))

    for p in test_paths:
        test_samples.append((p, label))

print(f"Train samples: {len(train_samples)}")
print(f"Test samples: {len(test_samples)}")

# ================= DATASET =================

class ImageDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        return torch.tensor(img), torch.tensor(label)

train_dataset = ImageDataset(train_samples)
test_dataset  = ImageDataset(test_samples)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_classes = len(classes)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID

                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):

                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            if patch.size == 0:
                                continue

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = [
                                    "mean_rel_low","mean_rel_mid","mean_rel_high",
                                    "std_rel_low","std_rel_high",
                                    "cv_low","cv_high",
                                    "max_rel_mean","max_rel_std"
                                ]

                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train_tm, y_train_tm = extract(train_loader)
X_test_tm, y_test_tm   = extract(test_loader)

print("Sanity check:", X_train_tm.dtype, np.unique(X_train_tm))

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train_tm, y_train_tm, epochs=1)

    preds = tm.predict(X_test_tm)
    acc = accuracy_score(y_test_tm, preds)

    history.append([epoch+1, acc])
    print(f"Epoch {epoch+1} → {acc:.4f}")

df = pd.DataFrame(history, columns=["epoch", "accuracy"])
df.to_excel("patch_4x4_results_sample.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ================= HEATMAP =================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_dataset[idx]
    img_np = img[0].cpu().numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test_tm[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape
    patch_h = h // PATCH_GRID
    patch_w = w // PATCH_GRID

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*patch_h:(i+1)*patch_h,
            j*patch_w:(j+1)*patch_w
        ]

        if patch.size == 0:
            continue

        patch_norm = patch / (patch.max() + 1e-6)
        weight = 2.0 if "max" in info["type"] else 1.0

        heatmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w] += patch_norm * weight

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)

    heatmap_norm = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    heatmap_color = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)

    cv2.imwrite(os.path.join(save_dir, f"heatmap_sample{idx}.png"), heatmap_color)
    print(f"✅ Heatmap saved: {idx}")

print("Generating heatmaps...")
for i in range(3):
    generate_heatmap(i)

Preparing dataset...
Train samples: 1154
Test samples: 191
Training CNN...
✅ CNN Done
Extracting features...
Sanity check: uint32 [0 1]
Training TM...
2026-04-14 07:02:49,874 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-14 07:02:49,876 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-14 07:02:49,877 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-14 07:02:49,878 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-14 07:02:49,879 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-14 07:02:50

In [1]:
# ==========================================================
# 🚀 EuroSAT → CNN → TM → INTERPRETATION (HEATMAP)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit

# ================= CONFIG =================

BASE_DIR = "EuroSAT"

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 64
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TM
CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= PREPARE DATA =================

print("Preparing datasets...")

train_samples = []
test_samples = []

classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    class_path = os.path.join(BASE_DIR, cname)

    if not os.path.isdir(class_path):
        continue

    paths = glob(os.path.join(class_path, "*"))
    random.shuffle(paths)

    train_paths = paths[:TRAIN_PER_CLASS]
    test_paths  = paths[TRAIN_PER_CLASS:]

    for p in train_paths:
        train_samples.append((p, label))

    for p in test_paths:
        test_samples.append((p, label))

print(f"Train samples: {len(train_samples)}")
print(f"Test samples: {len(test_samples)}")

# ================= DATASET =================

class EuroSATDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError(f"Error loading image: {path}")

        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        # HWC → CHW
        img = np.transpose(img, (2, 0, 1))

        return torch.tensor(img), torch.tensor(label)

# ================= LOADERS =================

train_loader = DataLoader(
    EuroSATDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    EuroSATDataset(test_samples),
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True
)

num_classes = len(classes)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    
                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID
                    
                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):
                    
                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = [
                                    "mean_rel_low","mean_rel_mid","mean_rel_high",
                                    "std_rel_low","std_rel_high",
                                    "cv_low","cv_high",
                                    "max_rel_mean","max_rel_std"
                                ]

                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

import pandas as pd

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)
    acc = accuracy_score(y_test, preds)

    history.append([epoch+1, acc])
    print(f"Epoch {epoch+1} → {acc:.4f}")

# SAVE TO EXCEL
df = pd.DataFrame(history, columns=["epoch", "accuracy"])
df.to_excel("patch_4x4_results_eurosat.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ==========================================================
# 🔥 INTERPRETATION (HEATMAP)
# ==========================================================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_dataset[idx]
    img_np = img[0].cpu().numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape
    patch_h = h // PATCH_GRID  # 16
    patch_w = w // PATCH_GRID  # 16

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*patch_h:(i+1)*patch_h,
            j*patch_w:(j+1)*patch_w
        ]
        if patch.size == 0:
            continue
        patch_norm = patch / (patch.max() + 1e-6)

        weight = 2.0 if "max" in info["type"] else 1.0

        # Use patch_h/patch_w instead of hardcoded 32
        heatmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w] += patch_norm * weight

    # rest unchanged ...

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)

    heatmap_norm = cv2.normalize(
        heatmap, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(
        heatmap_norm, cv2.COLORMAP_JET
    )

    cv2.imwrite(os.path.join(save_dir, f"heatmap_eurosat_{idx}.png"), heatmap_color)

    print(f"✅ Heatmap saved for sample {idx}")

# ================= RUN =================

print("Generating interpretation...")

for i in range(3):
    generate_heatmap(i)

Preparing datasets...
Train samples: 1210
Test samples: 25790
Training CNN...
✅ CNN Done
Extracting features...
Training TM...
2026-04-14 17:12:38,309 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-14 17:12:38,311 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-14 17:12:38,312 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-14 17:12:38,313 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-14 17:12:38,313 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-14 17:12:38,753 - tmu.clause_bank.c

NameError: name 'test_dataset' is not defined

In [1]:
# ==========================================================
# 🚀 EuroSAT → CNN → TM → INTERPRETATION (HEATMAP)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit

# ================= CONFIG =================

BASE_DIR = "EuroSAT"

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 64
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TM
CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= PREPARE DATA =================

print("Preparing datasets...")

train_samples = []
test_samples = []

classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    class_path = os.path.join(BASE_DIR, cname)

    if not os.path.isdir(class_path):
        continue

    paths = glob(os.path.join(class_path, "*"))
    random.shuffle(paths)

    train_paths = paths[:TRAIN_PER_CLASS]
    test_paths  = paths[TRAIN_PER_CLASS:]

    for p in train_paths:
        train_samples.append((p, label))

    for p in test_paths:
        test_samples.append((p, label))

print(f"Train samples: {len(train_samples)}")
print(f"Test samples: {len(test_samples)}")

# ================= DATASET =================

class EuroSATDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError(f"Error loading image: {path}")

        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        # HWC → CHW
        img = np.transpose(img, (2, 0, 1))

        return torch.tensor(img), torch.tensor(label)

# ================= LOADERS =================

train_loader = DataLoader(
    EuroSATDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    EuroSATDataset(test_samples),
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True
)

num_classes = len(classes)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    
                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID
                    
                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):
                    
                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = [
                                    "mean_rel_low","mean_rel_mid","mean_rel_high",
                                    "std_rel_low","std_rel_high",
                                    "cv_low","cv_high",
                                    "max_rel_mean","max_rel_std"
                                ]

                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

import pandas as pd

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)
    acc = accuracy_score(y_test, preds)

    history.append([epoch+1, acc])
    print(f"Epoch {epoch+1} → {acc:.4f}")

# SAVE TO EXCEL
df = pd.DataFrame(history, columns=["epoch", "accuracy"])
df.to_excel("patch_4x4_results_eurosat.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ==========================================================
# 🔥 INTERPRETATION (HEATMAP)
# ==========================================================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_loader.dataset[idx]   # ✅ FIXED
    img_np = img.mean(dim=0).cpu().numpy()  # ✅ better visualization

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape
    patch_h = h // PATCH_GRID
    patch_w = w // PATCH_GRID

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*patch_h:(i+1)*patch_h,
            j*patch_w:(j+1)*patch_w
        ]
        if patch.size == 0:
            continue
        patch_norm = patch / (patch.max() + 1e-6)

        weight = 2.0 if "max" in info["type"] else 1.0

        # Use patch_h/patch_w instead of hardcoded 32
        heatmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w] += patch_norm * weight

    # rest unchanged ...

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)

    heatmap_norm = cv2.normalize(
        heatmap, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(
        heatmap_norm, cv2.COLORMAP_JET
    )

    cv2.imwrite(os.path.join(save_dir, f"heatmap_eurosat_{idx}.png"), heatmap_color)

    print(f"✅ Heatmap saved for sample {idx}")

# ================= RUN =================

print("Generating interpretation...")

for i in range(3):
    generate_heatmap(i)

Preparing datasets...
Train samples: 1210
Test samples: 25790
Training CNN...
✅ CNN Done
Extracting features...
Training TM...
2026-04-15 05:32:19,112 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-15 05:32:19,114 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-15 05:32:19,115 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-15 05:32:19,116 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-15 05:32:19,118 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-15 05:32:19,668 - tmu.clause_bank.c

In [1]:
# ==========================================================
# 🚀 EuroSAT → CNN → TM → INTERPRETATION (HEATMAP)
# ==========================================================

import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from glob import glob
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from tmu.models.classification.vanilla_classifier import TMClassifier
import pycuda.autoinit
import pandas as pd

# ================= CONFIG =================

BASE_DIR = "EuroSAT"

IMAGE_SIZE = 64
TRAIN_PER_CLASS = 121

BATCH_SIZE = 64
CNN_EPOCHS = 40

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TM
CLAUSES = 4000
T = 2500
S = 6.0
TM_EPOCHS = 50

# ================= PREPARE DATA =================

print("Preparing datasets...")

train_samples = []
test_samples = []

classes = sorted(os.listdir(BASE_DIR))

for label, cname in enumerate(classes):
    class_path = os.path.join(BASE_DIR, cname)

    if not os.path.isdir(class_path):
        continue

    paths = glob(os.path.join(class_path, "*"))
    random.shuffle(paths)

    train_paths = paths[:TRAIN_PER_CLASS]
    test_paths  = paths[TRAIN_PER_CLASS:]

    for p in train_paths:
        train_samples.append((p, label))

    for p in test_paths:
        test_samples.append((p, label))

print(f"Train samples: {len(train_samples)}")
print(f"Test samples: {len(test_samples)}")

# ================= DATASET =================

class EuroSATDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError(f"Error loading image: {path}")

        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0

        img = np.transpose(img, (2, 0, 1))

        return torch.tensor(img), torch.tensor(label)

# ================= LOADERS =================

train_loader = DataLoader(
    EuroSATDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    EuroSATDataset(test_samples),
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True
)

num_classes = len(classes)

# ================= CNN =================

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = nn.Conv2d(3,16,3,padding=1)
        self.conv5 = nn.Conv2d(3,16,5,padding=2)
        self.conv7 = nn.Conv2d(3,16,7,padding=3)

        self.block1 = nn.Sequential(
            nn.Conv2d(48,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

    def forward(self,x):
        x = torch.cat([
            F.relu(self.conv3(x)),
            F.relu(self.conv5(x)),
            F.relu(self.conv7(x))
        ], dim=1)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x

model = CNN().to(DEVICE)
classifier = nn.Linear(256, num_classes).to(DEVICE)

# ================= TRAIN CNN =================

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("Training CNN...")

for epoch in range(CNN_EPOCHS):
    model.train()
    classifier.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        feat = model(x)
        pooled = F.adaptive_avg_pool2d(feat,1).view(x.size(0), -1)
        out = classifier(pooled)

        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

print("✅ CNN Done")

# ================= FEATURE EXTRACTION =================

feature_map_info = []
built = False

def extract(loader):
    global built
    X, y = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            feat = model(xb)

            batch_feats = []

            for sample in feat:
                sample_features = []
                ch_idx = 0

                for ch in sample:
                    ch_np = ch.cpu().numpy()

                    ch_mean = ch_np.mean()
                    ch_std  = ch_np.std() + 1e-6

                    PATCH_GRID = 4
                    h, w = ch_np.shape
                    patch_h = h // PATCH_GRID
                    patch_w = w // PATCH_GRID

                    for i in range(PATCH_GRID):
                        for j in range(PATCH_GRID):

                            patch = ch_np[
                                i*patch_h:(i+1)*patch_h,
                                j*patch_w:(j+1)*patch_w
                            ]

                            mean = patch.mean()
                            std  = patch.std()
                            mx   = patch.max()

                            mean_rel = mean / (ch_mean + 1e-6)
                            std_rel  = std / ch_std
                            cv       = std / (mean + 1e-6)

                            f = [
                                int(mean_rel > 0.8),
                                int(mean_rel > 1.0),
                                int(mean_rel > 1.2),
                                int(std_rel > 0.8),
                                int(std_rel > 1.0),
                                int(cv > 0.5),
                                int(cv > 1.0),
                                int(mx > ch_mean),
                                int(mx > ch_mean + ch_std)
                            ]

                            sample_features.extend(f)

                            if not built:
                                types = [
                                    "mean_rel_low","mean_rel_mid","mean_rel_high",
                                    "std_rel_low","std_rel_high",
                                    "cv_low","cv_high",
                                    "max_rel_mean","max_rel_std"
                                ]

                                for t in types:
                                    feature_map_info.append({
                                        "channel": ch_idx,
                                        "pos": (i,j),
                                        "type": t
                                    })

                    ch_idx += 1

                batch_feats.append(sample_features)

            built = True
            X.append(np.array(batch_feats))
            y.extend(yb.numpy())

    return np.vstack(X).astype(np.uint32), np.array(y, dtype=np.uint32)

print("Extracting features...")
X_train, y_train = extract(train_loader)
X_test, y_test   = extract(test_loader)

# ================= TM =================

tm = TMClassifier(
    number_of_clauses=CLAUSES,
    T=T,
    s=S,
    weighted_clauses=True,
    feature_negation=True,
    platform="CUDA"
)

print("Training TM...")

history = []

for epoch in range(TM_EPOCHS):
    tm.fit(X_train, y_train, epochs=1)
    preds = tm.predict(X_test)

    acc = accuracy_score(y_test, preds)
    history.append([epoch+1, acc])

    print(f"Epoch {epoch+1} → {acc:.4f}")

    # ✅ PRINT ONLY FINAL EPOCH
    if epoch == TM_EPOCHS - 1:
        print("\n🔥 Sample Predictions (First 10):")
        for i in range(10):
            print(
                f"Sample {i} → Predicted: {classes[preds[i]]} "
                f"| True: {classes[y_test[i]]}"
            )

# SAVE RESULTS
df = pd.DataFrame(history, columns=["epoch", "accuracy"])
df.to_excel("patch_4x4_results_eurosat.xlsx", index=False)

print("🔥 FINAL ACC:", acc)

# ================= HEATMAP =================

def generate_heatmap(idx=0, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    img, _ = test_loader.dataset[idx]
    img_np = img.mean(dim=0).cpu().numpy()

    heatmap = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)

    active = np.where(X_test[idx] == 1)[0]

    PATCH_GRID = 4
    h, w = img_np.shape
    patch_h = h // PATCH_GRID
    patch_w = w // PATCH_GRID

    for f in active:
        if f >= len(feature_map_info):
            continue

        info = feature_map_info[f]
        i, j = info["pos"]

        patch = img_np[
            i*patch_h:(i+1)*patch_h,
            j*patch_w:(j+1)*patch_w
        ]

        if patch.size == 0:
            continue

        patch_norm = patch / (patch.max() + 1e-6)
        weight = 2.0 if "max" in info["type"] else 1.0

        heatmap[i*patch_h:(i+1)*patch_h, j*patch_w:(j+1)*patch_w] += patch_norm * weight

    heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)

    heatmap_norm = cv2.normalize(
        heatmap, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)

    heatmap_color = cv2.applyColorMap(
        heatmap_norm, cv2.COLORMAP_JET
    )

    cv2.imwrite(os.path.join(save_dir, f"heatmap_eurosat_{idx}.png"), heatmap_color)

    print(f"✅ Heatmap saved for sample {idx}")

print("Generating interpretation...")classification

for i in range(3):
    generate_heatmap(i)

Preparing datasets...
Train samples: 1210
Test samples: 25790
Training CNN...
✅ CNN Done
Extracting features...
Training TM...
2026-04-22 02:23:18,520 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/3201a5ec836dac638412a932dacde82d83e549ad.ptx'.
2026-04-22 02:23:18,522 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/89016a311bdbf3c0a8b45a95c9a4083dcbd88f19.ptx'.
2026-04-22 02:23:18,524 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/339c1aa30de39fac8aa528e545297542c8ed6031.ptx'.
2026-04-22 02:23:18,525 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/d0add23ba02c3e6bf425e1e0f990c8c3b35e0030.ptx'.
2026-04-22 02:23:18,526 - tmu.clause_bank.clause_bank_cuda - INFO - Loading compiled CUDA module from '/tmp/tm_kernels/f71fb0670b8df2a0be62e147eaca127f46a1d7dd.ptx'.
2026-04-22 02:23:19,290 - tmu.clause_bank.c